# Citus Database Setup and SDK Demo (Single Node)

This notebook demonstrates how to set up a Citus database using a single-node docker container, populate the patch table with dummy data, and implement a Python SDK for interacting with the database. Worker node logic is omitted for single-node setup.

## 1. Install and Import Required Libraries

Install `psycopg` if not already installed, and import all required libraries for database interaction.

In [1]:
# Install psycopg if needed (uncomment if running in a new environment)
# !pip install psycopg[binary]

import psycopg
from psycopg.rows import dict_row
import random
import datetime
import base64
from db_client import CitusHeadClient
import os

In [2]:
# Import DB connection constants from constants.py
from constants import (
    CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD
)

In [3]:
# Set DB connection variables from constants (single-node)
DB_HOST = CITUS_HEAD_HOST
DB_PORT = CITUS_HEAD_PORT
DB_NAME = CITUS_HEAD_DB
DB_USER = CITUS_HEAD_USER
DB_PASSWORD = CITUS_HEAD_PASSWORD

In [4]:
NUM_PATCHES = 100

## 0. Drop All Tables (Clean Start)

Drop all tables if they exist to ensure a clean setup.

In [5]:
head_client = CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD)
head_client.drop_all_tables()
print("All tables dropped (if existed).")


All tables dropped (if existed).


## 2. Connect to Citus Node

Establish a connection to the Citus/Postgres node using psycopg. Store connection parameters securely (e.g., using environment variables).

In [6]:
# Use CitusHeadClient for connection
def get_head_connection():
    return CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD).get_connection()

# Test connection
with get_head_connection() as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT version();')
        print('Connected to:', cur.fetchone()['version'])

Connected to: PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 3. Create Database Schema (Tables)

Create all tables as described in the technical design document, including distributed and reference tables. Use Citus distribution commands where required.

In [7]:
head_client.setup_schema()
print("Schema and distribution setup complete.")


Schema and distribution setup complete.


In [8]:
head_client.setup_triggers()


INSERT trigger function created on coordinator and workers.
UPDATE trigger function created on coordinator and workers.
Per-shard INSERT triggers installed on pred_patch_latest shards.
Per-shard UPDATE triggers installed on patch shards.

All per-shard triggers installed.


## 4. Verify Table Creation

Query the information schema to verify that all tables have been created successfully.

In [9]:
# List all tables in the public schema
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        tables = [row['table_name'] for row in cur.fetchall()]
        print('Tables in public schema:', tables)

Tables in public schema: ['citus_schemas', 'citus_tables', 'confusion_matrix_ln', 'image', 'label_class', 'patch', 'pred_patch_last', 'pred_patch_latest', 'project', 'settings']


## 5. Insert Dummy Data into Patch Table

Generate and insert dummy data into the patch table, ensuring all required fields are populated and constraints are respected.

In [10]:
# Helper: Insert dummy project, image, label_class for FK constraints
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("INSERT INTO project (project_name, description) VALUES (%s, %s) RETURNING project_id;", ('Demo Project', 'For dummy data'))
        project_id = cur.fetchone()['project_id']
        cur.execute("INSERT INTO image (project_id, name, image_path, upload_ts, base_mag, base_width, base_height, deepzoom_tilesize) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) RETURNING image_id;",
                    (project_id, 'Demo Image', '/tmp/demo.tif', datetime.datetime.now(), 20.0, 10000, 8000, 256))
        image_id = cur.fetchone()['image_id']
        cur.execute("INSERT INTO label_class (project_id, name, color_code, event_ts) VALUES (%s, %s, %s, %s) RETURNING label_class_id;",
                    (project_id, 'Tumor', '#FF0000', datetime.datetime.now()))
        label_class_id = cur.fetchone()['label_class_id']
        print(f"Inserted project_id={project_id}, image_id={image_id}, label_class_id={label_class_id}")

def random_bytes(size=128):
    return os.urandom(size)

for i in range(NUM_PATCHES):
    patch_id = head_client.insert_patch(1000 + i, label_class_id, image_id, 20.0, random_bytes())
    print(f"Inserted patch_id={patch_id}")

Inserted project_id=1, image_id=1, label_class_id=1
Inserted patch_id=1
Inserted patch_id=2
Inserted patch_id=3
Inserted patch_id=4
Inserted patch_id=5
Inserted patch_id=6
Inserted patch_id=7
Inserted patch_id=8
Inserted patch_id=9
Inserted patch_id=10
Inserted patch_id=11
Inserted patch_id=12
Inserted patch_id=13
Inserted patch_id=14
Inserted patch_id=15
Inserted patch_id=16
Inserted patch_id=17
Inserted patch_id=18
Inserted patch_id=19
Inserted patch_id=20
Inserted patch_id=21
Inserted patch_id=22
Inserted patch_id=23
Inserted patch_id=24
Inserted patch_id=25
Inserted patch_id=26
Inserted patch_id=27
Inserted patch_id=28
Inserted patch_id=29
Inserted patch_id=30
Inserted patch_id=31
Inserted patch_id=32
Inserted patch_id=33
Inserted patch_id=34
Inserted patch_id=35
Inserted patch_id=36
Inserted patch_id=37
Inserted patch_id=38
Inserted patch_id=39
Inserted patch_id=40
Inserted patch_id=41
Inserted patch_id=42
Inserted patch_id=43
Inserted patch_id=44
Inserted patch_id=45
Inserted pat

In [11]:
# Check number of shards for the patch table and print row counts per shard, including empty shards
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        # Number of shards
        cur.execute("SELECT count(*) FROM pg_dist_shard WHERE logicalrelid = 'patch'::regclass;")
        num_shards = cur.fetchone()['count']
        print(f"Number of shards for 'patch' table: {num_shards}")
        # Row counts per shard, including empty
        cur.execute("""
            SELECT s.shardid, COALESCE(count(p.patch_id), 0) as row_count
            FROM pg_dist_shard s
            LEFT JOIN patch p ON get_shard_id_for_distribution_column('patch', p.patch_id) = s.shardid
            WHERE s.logicalrelid = 'patch'::regclass
            GROUP BY s.shardid
            ORDER BY s.shardid;
        """)
        rows = cur.fetchall()
        empty_count = 0
        for row in rows:
            print(f"Shard {row['shardid']}: {row['row_count']} rows")
            if row['row_count'] == 0:
                empty_count += 1
        print(f"Empty shards: {empty_count} out of {num_shards}")

Number of shards for 'patch' table: 32
Shard 110370: 3 rows
Shard 110371: 5 rows
Shard 110372: 2 rows
Shard 110373: 6 rows
Shard 110374: 3 rows
Shard 110375: 1 rows
Shard 110376: 2 rows
Shard 110377: 2 rows
Shard 110378: 6 rows
Shard 110379: 2 rows
Shard 110380: 2 rows
Shard 110381: 2 rows
Shard 110382: 3 rows
Shard 110383: 3 rows
Shard 110384: 5 rows
Shard 110385: 2 rows
Shard 110386: 3 rows
Shard 110387: 2 rows
Shard 110388: 0 rows
Shard 110389: 3 rows
Shard 110390: 6 rows
Shard 110391: 3 rows
Shard 110392: 3 rows
Shard 110393: 3 rows
Shard 110394: 3 rows
Shard 110395: 2 rows
Shard 110396: 1 rows
Shard 110397: 2 rows
Shard 110398: 7 rows
Shard 110399: 3 rows
Shard 110400: 5 rows
Shard 110401: 5 rows
Empty shards: 1 out of 32


## 6. Verify Dummy Data in Patch Table

Query the patch table to confirm that dummy data has been inserted correctly.

In [12]:
# Query and display dummy patch data
for row in head_client.fetch_patches(limit=10):
    print(row)

{'patch_id': 8, 'patch_uid': 1007, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'\\\xa2\xb6\xbf\x93\x08\x00\x88v21;\xde\x1b\xf6+\x07\x7f~zQ\x92\x99\xb2\xa7\xae\xd3\x98Aft\x14\xc5\x9f\x9d3\x17\xf0]i\x06\x91\xd0o\xabK=1\xe7*\xe7=\x15\x94Y\x81\xbclM=\xfc\r\xf5\xfd-\xd0r\xbeB\xac!Nv\x10\x01Q\xb5\x0f\xdb9\xcc\x93 \xd5DE\xb2\x0f\xaaU\x89\xcdYx\xff7\xb4\xe3n\x08\x8aga\xd9S\xb0\xb8\xc7D2\xcfMl\xbc\x8aF\x89J\xb1\x973\xaa5\xa4[\x8c\xfe"'}
{'patch_id': 20, 'patch_uid': 1019, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'wjd1\x00\x0e\xe4\xd68D\xf77h]\xb0\x04\xd4\xf2\xff`0\xc2?\x05\xca\xadx\xb3\xf3\xab#\x80N\xd9`\x7f\x9c\\9Q\x12\xad\xfeg\x86&\xa4\xd4\xe4\x00\xb0p\xd3\x10\x05\x14\xb4H\x88\xe7\x9fO\xdc\x1b8\x1b\x8a9\x97\x01\x8e\xd9q5\x11\xf56%\x8d\xfb\x1aK\xaf\xe5\xa8\xa3}*\x1a\xc4\xdd3\xf6}\x11AS$\xec\xef>M\xa1\xaf\x1e\xa7_S\xa7\x13\x8d\xb6\xa6y\xbd\x80\x08\x9e+\x1f\x0b3f\xac\xed\xac\xec\xb3'}
{'patch_id': 60, 'patch_uid': 1059, 'label_class_id

## 7. Implement db_client SDK: Head Node Level

Write Python classes and functions in `db_client.py` to interact with the database at the Citus head node level, including connection management and basic CRUD operations.

In [13]:
# db_client.py will be implemented in the next step.
# Example usage for SDK will be shown after SDK implementation.

In [14]:
# Example: Using db_client SDK (single-node)
# Uses constants.py for all connection parameters
head_client = CitusHeadClient()
print('Patches:', head_client.fetch_patches(limit=3))

# Insert a new patch (dummy data)
# patch_id = head_client.insert_patch(2000, 1, 1, 20.0, b'dummybytes')
# print('Inserted patch_id:', patch_id)

Patches: [{'patch_id': 8, 'patch_uid': 1007, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'\\\xa2\xb6\xbf\x93\x08\x00\x88v21;\xde\x1b\xf6+\x07\x7f~zQ\x92\x99\xb2\xa7\xae\xd3\x98Aft\x14\xc5\x9f\x9d3\x17\xf0]i\x06\x91\xd0o\xabK=1\xe7*\xe7=\x15\x94Y\x81\xbclM=\xfc\r\xf5\xfd-\xd0r\xbeB\xac!Nv\x10\x01Q\xb5\x0f\xdb9\xcc\x93 \xd5DE\xb2\x0f\xaaU\x89\xcdYx\xff7\xb4\xe3n\x08\x8aga\xd9S\xb0\xb8\xc7D2\xcfMl\xbc\x8aF\x89J\xb1\x973\xaa5\xa4[\x8c\xfe"'}, {'patch_id': 20, 'patch_uid': 1019, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'wjd1\x00\x0e\xe4\xd68D\xf77h]\xb0\x04\xd4\xf2\xff`0\xc2?\x05\xca\xadx\xb3\xf3\xab#\x80N\xd9`\x7f\x9c\\9Q\x12\xad\xfeg\x86&\xa4\xd4\xe4\x00\xb0p\xd3\x10\x05\x14\xb4H\x88\xe7\x9fO\xdc\x1b8\x1b\x8a9\x97\x01\x8e\xd9q5\x11\xf56%\x8d\xfb\x1aK\xaf\xe5\xa8\xa3}*\x1a\xc4\xdd3\xf6}\x11AS$\xec\xef>M\xa1\xaf\x1e\xa7_S\xa7\x13\x8d\xb6\xa6y\xbd\x80\x08\x9e+\x1f\x0b3f\xac\xed\xac\xec\xb3'}, {'patch_id': 60, 'patch_uid': 1059, 'la

<!-- Worker node logic omitted for single-node setup -->